# Bankai Fast MCS: 4,800 model empirical benchmark

This notebook benchmarks Bankai Fast MCS against the academic Python implementation using precomputed multivariate ARCH losses released with [Barde's paper](https://doi.org/10.1016/j.jeconom.2025.106123). It uses the paper's low-volatility sample, `rv5` proxy, one-day horizon, 1,000 block-bootstrap replications, block length 10, and two-pass algorithm.

The paper describes 4,800 candidate specifications. Failed or invalid forecasts are excluded from its loss matrices; the selected loss file contains **4,518 valid model columns**. Both counts are reported below. The timing covers MCS computation only, excluding download and deserialization. The official ZIP is 1.25 GB, but the range reader fetches only the selected loss file (about 90 MB). Peak working memory is several hundred MB.

The loss archive contains a Python pickle created by the paper authors. Only run this notebook against the pinned official release URL below.


In [1]:
%pip install -q bankai-fast-mcs pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.0/405.0 kB 15.2 MB/s eta 0:00:00


## Select the paper's empirical setting

Sample 1 is the low-volatility period. The author release stores losses for all five forecast horizons and both volatility proxies; these settings select the first horizon and `rv5` proxy. Change the constants to examine another published setting.


In [12]:
import hashlib
import importlib.util
import io
import pickle
import platform
import statistics
import time
import urllib.request
import zipfile
import zlib
import os
from importlib.metadata import version

import numpy as np
from bankai_fast_mcs import ModelConfidenceSet

SAMPLE = 1                 # 0 = high volatility; 1 = low volatility
PROXY = "rv5"
HORIZON = 1                # Forecast horizon in days (1 to 5)
BOOTSTRAP = "block"        # The paper also reports stationary bootstrap results
B = 1_000
BLOCK_LENGTH = 10
SEED = 0
ALGORITHM = "2-pass"
REPEATS = 1                # Increase for more stable timing at higher runtime cost

assert SAMPLE in (0, 1) and 1 <= HORIZON <= 5
assert PROXY in ("rv5", "open_to_close")


print(f"Python: {platform.python_version()}")
print(f"Bankai: {version('bankai-fast-mcs')}")

print(f"NumPy: {np.__version__}")
print(f"Logical CPUs: {os.cpu_count()} RAM: {round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024. ** 3), 2)} GB")


Python: 3.13.15
Bankai: 0.1.0a2
NumPy: 2.1.3
Logical CPUs: 44 RAM: 172.95 GB


## Load the published loss matrix

HTTP byte ranges read the ZIP directory and fetch one member, avoiding a full 1.25 GB download. If your network blocks range requests, download `losses.zip` from the [v1.0.0 replication release](https://github.com/Sylvain-Barde/mArch_mcs_analysis/releases/tag/v1.0.0) and read the matching member locally instead.


In [3]:
class HTTPRangeReader:
    """Small seekable reader for ZIP files on range-enabled HTTP servers."""

    def __init__(self, url):
        self.url = url
        request = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(request, timeout=60) as response:
            self.size = int(response.headers["Content-Length"])
        self.position = 0
        self.closed = False

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.position

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            position = offset
        elif whence == io.SEEK_CUR:
            position = self.position + offset
        elif whence == io.SEEK_END:
            position = self.size + offset
        else:
            raise ValueError(f"invalid whence: {whence}")
        if position < 0:
            raise ValueError("negative seek position")
        self.position = position
        return position

    def read(self, size=-1):
        if size is None or size < 0:
            size = self.size - self.position
        size = min(size, self.size - self.position)
        if size <= 0:
            return b""
        start, end = self.position, self.position + size - 1
        request = urllib.request.Request(
            self.url,
            headers={"Range": f"bytes={start}-{end}", "Accept-Encoding": "identity"},
        )
        with urllib.request.urlopen(request, timeout=120) as response:
            if response.status != 206:
                raise RuntimeError("The server did not honor HTTP range requests.")
            data = response.read()
        if len(data) != size:
            raise IOError(f"Expected {size} bytes, received {len(data)}.")
        self.position += len(data)
        return data

    def close(self):
        self.closed = True


LOSS_ARCHIVE_URL = (
    "https://github.com/Sylvain-Barde/mArch_mcs_analysis/releases/"
    "download/v1.0.0/losses.zip"
)
member_name = f"sample_{SAMPLE}_proxy_{PROXY}.pkl"

with zipfile.ZipFile(HTTPRangeReader(LOSS_ARCHIVE_URL)) as archive:
    member_info = archive.getinfo(member_name)
    print(f"Fetching {member_name} ({member_info.file_size / 1e6:.1f} MB)")
    compressed_pickle = archive.read(member_info)

loss_data = pickle.loads(zlib.decompress(compressed_pickle), encoding="bytes")
if isinstance(loss_data, dict):
    loss_data = {
        key.decode() if isinstance(key, bytes) else key: value
        for key, value in loss_data.items()
    }
losses = np.asarray(loss_data["losses"][:, :, HORIZON - 1], dtype=np.float64)

candidate_model_count = len(loss_data["univarType"])
valid_model_count = losses.shape[1]
print(f"Candidate specifications in paper: {candidate_model_count:,}")
print(f"Valid models in this loss matrix:   {valid_model_count:,}")
print(f"Loss matrix shape (observations, models): {losses.shape}")
print(f"Loss matrix memory: {losses.nbytes / 1e6:.1f} MB")
assert losses.shape[0] == 550


Fetching sample_1_proxy_rv5.pkl (94.0 MB)
Candidate specifications in paper: 4,800
Valid models in this loss matrix:   4,518
Loss matrix shape (observations, models): (550, 4518)
Loss matrix memory: 19.9 MB


## Run Bankai Fast MCS and the academic Python reference

Both implementations use the same losses, seed, bootstrap type, replication count, and block length. The Python reference source is pinned to the author's `v1.0.0` replication release. Timings include bootstrap index generation and MCS computation, but exclude imports and data loading.


In [4]:
reference_url = (
    "https://raw.githubusercontent.com/Sylvain-Barde/"
    "mArch_mcs_analysis/v1.0.0/fastMCS.py"
)
with urllib.request.urlopen(reference_url, timeout=60) as response:
    reference_source = response.read()
reference_sha256 = hashlib.sha256(reference_source).hexdigest()
expected_reference_sha256 = "9c68ee8eb5687924e477dec9baf18d9d91c9befd38a13244976a910882a1c9c0"
assert reference_sha256 == expected_reference_sha256, "Academic reference source hash changed."
reference_path = "/tmp/academic_fastMCS.py"
with open(reference_path, "wb") as file:
    file.write(reference_source)
spec = importlib.util.spec_from_file_location("academic_fastMCS", reference_path)
academic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(academic)

def run_bankai():
    model_set = ModelConfidenceSet(losses, seed=SEED, verbose=False)
    model_set.run(B=B, b=BLOCK_LENGTH, bootstrap=BOOTSTRAP, algorithm=ALGORITHM)
    return model_set

def run_academic():
    model_set = academic.mcs(seed=SEED, verbose=False)
    model_set.addLosses(losses)
    model_set.run(B=B, b=BLOCK_LENGTH, bootstrap=BOOTSTRAP, algorithm=ALGORITHM)
    return model_set

bankai_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    bankai_result = run_bankai()
    bankai_times.append(time.perf_counter() - start)

academic_times = []
for _ in range(REPEATS):
    start = time.perf_counter()
    academic_result = run_academic()
    academic_times.append(time.perf_counter() - start)

bankai_seconds = statistics.median(bankai_times)
academic_seconds = statistics.median(academic_times)
speedup = academic_seconds / bankai_seconds

print(f"Bankai Fast MCS median: {bankai_seconds:.3f} s")
print(f"Academic Python median: {academic_seconds:.3f} s")
print(f"Speedup: {speedup:.2f}x")


Bankai Fast MCS median: 36.582 s
Academic Python median: 177.135 s
Speedup: 4.84x


## Check statistical output parity

The score vectors and bootstrap distributions are compared within floating-point tolerance. Model orders are compared by their ranked score values as well as by exact model index. Near-equal scores can sort differently across NumPy and Rust; the cell reports those rank and p-value differences, and checks MCS membership without depending on output order.


In [ ]:
bankai_order = np.asarray(bankai_result.elimination_order, dtype=np.int64)
academic_order = np.asarray(academic_result.exclMods, dtype=np.int64)

np.testing.assert_allclose(
    bankai_result.t_score, academic_result.tScore, rtol=1e-9, atol=1e-10
)
np.testing.assert_allclose(
    bankai_result.t_boot_distribution,
    academic_result.tBootDist,
    rtol=1e-9,
    atol=1e-10,
)

# Compare score values in rank order. Distinct implementations may order
# numerically tied scores differently even when the ranked score vectors match.
bankai_ranked_scores = bankai_result.t_score[bankai_order]
academic_ranked_scores = academic_result.tScore[academic_order]
np.testing.assert_allclose(
    bankai_ranked_scores, academic_ranked_scores, rtol=1e-9, atol=1e-10
)
rank_index_differences = int(np.count_nonzero(bankai_order != academic_order))

bankai_included, bankai_excluded = bankai_result.get_mcs(alpha=0.1)
academic_included, academic_excluded = academic_result.getMCS(alpha=0.1)
bankai_included = np.asarray(bankai_included, dtype=np.int64)
bankai_excluded = np.asarray(bankai_excluded, dtype=np.int64)
academic_included = np.asarray(academic_included, dtype=np.int64)
academic_excluded = np.asarray(academic_excluded, dtype=np.int64)

# P-values are indexed in elimination order by both APIs; map them back to
# model IDs before comparing them.
bankai_p_by_model = np.empty(valid_model_count, dtype=np.float64)
academic_p_by_model = np.empty(valid_model_count, dtype=np.float64)
bankai_p_by_model[bankai_order] = bankai_result.p_values
academic_p_by_model[academic_order] = academic_result.pVals
p_value_differences = np.abs(bankai_p_by_model - academic_p_by_model)

mcs_membership_differences = int(
    np.setxor1d(bankai_included, academic_included).size
)
print(f"Score vectors and bootstrap distributions match within tolerance.")
print(
    f"Different model indices in elimination order: "
    f"{rank_index_differences:,} / {valid_model_count:,}"
)
print(
    f"P-values differing by model (>1e-12): "
    f"{np.count_nonzero(p_value_differences > 1e-12):,}; "
    f"maximum difference: {p_value_differences.max():.6g}"
)
print(f"MCS membership differences at alpha=0.10: {mcs_membership_differences:,}")
print(
    f"MCS sizes (Bankai / academic): "
    f"{bankai_included.size:,} / {academic_included.size:,}"
)


## References

- Barde, S. (2025). [Large-scale model comparison with fast model confidence sets](https://doi.org/10.1016/j.jeconom.2025.106123). *Journal of Econometrics*.
- [Author's replication repository and v1.0.0 release](https://github.com/Sylvain-Barde/mArch_mcs_analysis/releases/tag/v1.0.0). The selected loss file comes from `losses.zip`; invalid forecasts are documented in the archive metadata.
- [Academic fastMCS source](https://github.com/Sylvain-Barde/mArch_mcs_analysis/blob/v1.0.0/fastMCS.py).
